In [1]:
# ============================================================================
# EEG Classification Pipeline 
# ============================================================================
import sys
sys.path.append("/teamspace/studios/this_studio/Phd")
from src.tools import tools
from config import config
# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    modules = tools()
    modules.set_seed(seed=modules.config['random_state'])
    modules.print_config()
    modules.setup_output_directory()
    
    try:
        print("\nGet dataset...")
        folds = modules.dataset.data_split(
            k_folds=modules.config['k_folds'], 
            groups=modules.config['groups'],
            fold_type=modules.config['fold_strategy'],
            strategy=modules.config['strategy'],
            val_split=modules.config['val_split'],
            random_state=modules.config['random_state']
        )
        print(f"\nCreated {len(folds)} folds")
        print(f"\n{'='*80}\nStarting {modules.config['k_folds']}-fold cross-validation\n{'='*80}")
        
        all_fold_results = []
        successful_folds = 0
        
        for fold_config in folds:
            modules.set_seed(seed=modules.config['random_state'])
            result = modules.run_single_fold(fold_config)
            if result is not None:
                modules.save_segment_details(result)
                all_fold_results.append(result)
                successful_folds += 1
                print(f"\nCompleted {successful_folds}/{len(folds)} folds")
                
        modules.save_all_fold_details(all_fold_results)
        #=======================================================================
        if all_fold_results:
            modules.print_final_results(all_fold_results)
            modules.save_subject_voting_proportions(all_fold_results)
            print(f"\n{'='*80}\nPIPELINE COMPLETED SUCCESSFULLY!\n{'='*80}")
            print(f"Successful folds: {successful_folds}/{len(folds)}")
            
            #=========================================
            modules.generate_plots_from_saved_results()
            modules.save_segment_csvs_from_results()
            t = modules.save_subject_voting_proportions(all_fold_results)
            modules.report_performance_separately(t)
           
        else:
            print(f"\n{'!'*80}\nPIPELINE FAILED - No folds completed successfully\n{'!'*80}")

    except Exception as e:
        print(f"\n{'!'*80}\nPIPELINE FAILED: {e}\n{'!'*80}")
        traceback.print_exc()
    
    finally:
        modules.clear_memory()
        print("\nMemory cleanup completed")   


if __name__ == "__main__":
    main()

Loading information from: /teamspace/studios/this_studio/phdResearch/data1/Labels/labels.csv
Contains 65 Subjects

EEG CLASSIFICATION PIPELINE
Mode: Train-Test Split (No Validation)
Configuration:
--------------------------------------------------------------------------------
  Dataset path:      /teamspace/studios/this_studio/phdResearch/data1/
  Device:            cuda
  Model choice:      0
  Fold strategy:     manual65
  Number of folds:   5
  Validation split:  None
  Early stopping:    Disabled

Stratification: on (['Group'])

Data Settings:
  Class names:       ['HC', 'AD']
  Normalization:     none
  Norm level:        per_channel

Training Settings:
  Batch size:        16
  Learning rate:     0.001
  Max epochs:        30

Evaluation Settings:
  Aggregation mode:  majority
  Confidence filter: 0.7
  Save models:       True
Existing experiment directory deleted: /teamspace/studios/this_studio/Phd/experiment/AHEPA/results/Ahepagithub

Output directory setup complete: /teamspac